# Constrained-$N_J$ ratio squeezing scan

Analyze five two-group MCWF runs with fixed $N_i=(50,50)$ and couplings $(\omega_1,\omega_2)=(0.7,1.3)$. The initial sectors obey $N_{J1}=A N_{J2}$ for $A=(1,1.5,2,3,4)$, with `joint_probability=0.6` as a cap relative to the complete unconstrained wavefunction. The rational value $A=1.5$ is imposed exactly as $2N_{J1}-3N_{J2}=0$. Each constraint line contains less than 60% of the full probability, so all eligible sectors on that line are retained.

In [ ]:
from pathlib import Path
import sys

import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "common").is_dir() and (candidate / "slurm").is_dir():
            return candidate
    raise RuntimeError("Could not find the repository root.")


repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

analysis_dir = (
    repo_root / "slurm" / "analyses" / "squeezing_nj_ratio-scan"
)
input_dir = analysis_dir / "output"

from common.plotting import plot_bloch_angles, plot_spin_components, plot_squeezing
from parser.mcwf import TrajectoryEnsemble
from parser.moments import MomentSeries, SimulationMetadata
from post_analysis import (
    build_dressed_basis_transformation,
    compute_gell_mann_rotation,
    compute_j_moments,
    compute_mean_generator_moments,
    compute_s_moments,
    compute_squeezing,
    rotate_generator_moments,
)
from slurm.trajectory_ensemble_io import load_trajectory_ensemble_artifact

%reload_ext autoreload
%autoreload 2

## Input artifacts

Set `run_name` to the Slurm job name used by `submit.slurm`. Each value of $A$ corresponds to one array-task suffix.

In [ ]:
run_name = "nj-ratio-v1"
A_values = (1, 1.5, 2, 3, 4)
constraint_rows = ((1, -1), (2, -3), (1, -2), (1, -3), (1, -4))
filename_A_values = ("1", "1.5", "2", "3", "4")

artifact_paths = {
    A: input_dir / f"{run_name}_A_{filename_A}_{task_index}.pkl"
    for task_index, (A, filename_A) in enumerate(
        zip(A_values, filename_A_values)
    )
}
missing_paths = tuple(
    path for path in artifact_paths.values() if not path.is_file()
)
if missing_paths:
    raise FileNotFoundError(f"Missing simulation artifacts: {missing_paths}")

artifact_paths

## Generic moment and squeezing analysis

Process one artifact at a time. Average complete trajectory rows first, construct J and S, rotate the averaged moments into the dressed frame, and compute group-resolved and full-system squeezing. The trajectory-resolved arrays are released after each scan point.

In [ ]:
moments_by_A = {}
ntraj_by_A = {}
sectors_by_A = {}

for task_index, (A, constraint_row) in enumerate(
    zip(A_values, constraint_rows)
):
    artifact_path = artifact_paths[A]
    artifact = load_trajectory_ensemble_artifact(artifact_path)

    required_fields = {
        "ensemble", "simulation_metadata", "array_index", "ntraj"
    }
    missing_fields = required_fields.difference(artifact)
    if missing_fields:
        raise ValueError(
            f"{artifact_path} is missing fields: {sorted(missing_fields)}"
        )

    ensemble = artifact["ensemble"]
    metadata = artifact["simulation_metadata"]
    if not isinstance(ensemble, TrajectoryEnsemble):
        raise TypeError(f"{artifact_path} does not contain a TrajectoryEnsemble.")
    if not isinstance(metadata, SimulationMetadata):
        raise TypeError(f"{artifact_path} does not contain SimulationMetadata.")
    if artifact["array_index"] != task_index:
        raise ValueError(f"{artifact_path} has the wrong array-task index.")
    if artifact["ntraj"] != ensemble.moments.num_trajectories:
        raise ValueError(f"{artifact_path} has inconsistent trajectory counts.")
    if ensemble.metadata is None:
        raise ValueError(f"{artifact_path} has no ensemble metadata.")
    if metadata.Ni != (50, 50):
        raise ValueError(f"{artifact_path} has the wrong physical group sizes.")
    if not np.allclose(metadata.omega_groups, (0.7, 1.3)):
        raise ValueError(f"{artifact_path} has the wrong group couplings.")
    if any(
        len(sector) != 2
        or sum(
            coefficient * value
            for coefficient, value in zip(constraint_row, sector)
        ) != 0
        for sector in ensemble.metadata.sectors
    ):
        raise ValueError(f"{artifact_path} violates NJ1 = A * NJ2.")

    raw_moments = ensemble.moments
    moments = MomentSeries(t=raw_moments.t.copy(), metadata=metadata)
    moments.mean_generator_moments = compute_mean_generator_moments(
        raw_moments
    )
    moments.J = compute_j_moments(
        moments.mean_generator_moments,
        metadata=moments.metadata,
    )
    moments.S = compute_s_moments(
        moments.mean_generator_moments,
        moments.J,
    )

    transformation = build_dressed_basis_transformation(moments.J, moments.S)
    rotation = compute_gell_mann_rotation(transformation)
    mean_dressed_generator_moments = rotate_generator_moments(
        moments.mean_generator_moments,
        rotation,
    )
    moments.squeezing = compute_squeezing(
        mean_dressed_generator_moments,
        moments.J,
        metadata=moments.metadata,
        include_full_system=True,
    )

    moments_by_A[A] = moments
    ntraj_by_A[A] = raw_moments.num_trajectories
    sectors_by_A[A] = tuple(ensemble.metadata.sectors)
    print(
        f"A={A}: {ntraj_by_A[A]} trajectories, "
        f"sectors={sectors_by_A[A]}, couplings={metadata.omega_groups}"
    )

    del artifact, ensemble, raw_moments

reference = moments_by_A[A_values[0]]
for A in A_values[1:]:
    moments = moments_by_A[A]
    if not np.allclose(moments.t, reference.t, atol=1e-12, rtol=0.0):
        raise ValueError("All scan points must share the same saved-time grid.")
    if moments.metadata.phase_protocol != reference.metadata.phase_protocol:
        raise ValueError("All scan points must share the same phase protocol.")

## Full-system squeezing

In [ ]:
full_system_squeezing = tuple(
    moments_by_A[A].squeezing.full_system for A in A_values
)
if any(series is None for series in full_system_squeezing):
    raise RuntimeError("Full-system squeezing was not computed.")

full_squeezing_fig, full_squeezing_axes = plot_squeezing(
    reference.t,
    tuple(series.xi2 for series in full_system_squeezing),
    tuple(series.lambda_min for series in full_system_squeezing),
    tuple(series.N_c for series in full_system_squeezing),
    tuple(series.excited_fraction_active for series in full_system_squeezing),
    labels=tuple(rf"$A={A}$" for A in A_values),
    shade_index=3,
    phase_protocol=reference.metadata.phase_protocol,
    title="Full-system generalized squeezing",
)

## Group-resolved squeezing

Color identifies $A$; solid and dashed curves identify groups 1 and 2.

In [ ]:
group_squeezing_axes = None
group_linestyles = ("-", "--")

for colour_family_index, A in enumerate(A_values):
    for group_index, linestyle in enumerate(group_linestyles):
        series = moments_by_A[A].squeezing.group_resolved[group_index]
        group_squeezing_fig, group_squeezing_axes = plot_squeezing(
            series.t,
            series.xi2,
            series.lambda_min,
            series.N_c,
            series.excited_fraction_active,
            labels=(rf"$A={A}$, group {group_index + 1}",),
            colour_family_index=colour_family_index,
            shade_index=3,
            linestyle=linestyle,
            axes=group_squeezing_axes,
            phase_protocol=reference.metadata.phase_protocol,
            title="Group-resolved generalized squeezing",
        )

## Aggregated J angles and components

In [ ]:
scan_labels = tuple(rf"$A={A}$" for A in A_values)

aggregated_angle_fig, aggregated_angle_axes = plot_bloch_angles(
    reference.t,
    tuple(moments_by_A[A].J.aggregated.theta for A in A_values),
    tuple(moments_by_A[A].J.aggregated.phi for A in A_values),
    labels=scan_labels,
    shade_index=3,
    phase_protocol=reference.metadata.phase_protocol,
    title="Aggregated J-sphere angles",
)

aggregated_component_fig, aggregated_component_axes = plot_spin_components(
    reference.t,
    tuple(moments_by_A[A].J.aggregated.x for A in A_values),
    tuple(moments_by_A[A].J.aggregated.y for A in A_values),
    tuple(moments_by_A[A].J.aggregated.z for A in A_values),
    tuple(moments_by_A[A].J.aggregated.length for A in A_values),
    labels=scan_labels,
    shade_index=3,
    phase_protocol=reference.metadata.phase_protocol,
    title="Aggregated J-sphere components",
)

## Group-resolved J angles and components

Use the same color and linestyle convention as the group-resolved squeezing plot.

In [ ]:
group_angle_axes = None
group_component_axes = None

for colour_family_index, A in enumerate(A_values):
    J = moments_by_A[A].J
    for group_index, linestyle in enumerate(group_linestyles):
        label = (rf"$A={A}$, group {group_index + 1}",)
        group_angle_fig, group_angle_axes = plot_bloch_angles(
            J.t,
            J.theta[group_index],
            J.phi[group_index],
            labels=label,
            colour_family_index=colour_family_index,
            shade_index=3,
            linestyle=linestyle,
            axes=group_angle_axes,
            phase_protocol=reference.metadata.phase_protocol,
            title="Group-resolved J-sphere angles",
        )
        group_component_fig, group_component_axes = plot_spin_components(
            J.t,
            J.x[group_index],
            J.y[group_index],
            J.z[group_index],
            J.length[group_index],
            labels=label,
            colour_family_index=colour_family_index,
            shade_index=3,
            linestyle=linestyle,
            axes=group_component_axes,
            phase_protocol=reference.metadata.phase_protocol,
            title="Group-resolved J-sphere components",
        )

## Analysis summary

In [ ]:
{
    A: {
        "ntraj": ntraj_by_A[A],
        "sectors": sectors_by_A[A],
        "couplings": moments_by_A[A].metadata.omega_groups,
        "minimum group xi2 [dB]": tuple(
            float(np.nanmin(series.xi2_db))
            for series in moments_by_A[A].squeezing.group_resolved
        ),
        "minimum full-system xi2 [dB]": float(
            np.nanmin(moments_by_A[A].squeezing.full_system.xi2_db)
        ),
    }
    for A in A_values
}